# 03 — Validate the frozen unanswerable set

A retrieval system should decline questions when the corpus has no answer. We created 30 synthetic negatives before improving BM25: ten missing agreements, ten unsupported provisions, and ten subsequent business events.

A negative label means **the corpus cannot establish the requested answer**, not that the real-world answer is no. Each question has a rationale and supporting absence checks. Literal searches are useful checks, not semantic proof; these labels are not expert legal annotations.

The prescribed hash split gives 23 train and 7 test questions. We retain this exact assignment rather than selecting questions to obtain a preferred balance. Only train negatives are retrieved below. The full set must be committed before this notebook measures abstention.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from retrieval import BM25Retriever, load_inputs
from negative_validation import validate_negatives, evaluate_negative_train

_, documents, _ = load_inputs(ROOT)
records, frozen = validate_negatives(ROOT, documents)
print('Frozen questions:', len(records))
print('Split counts:', frozen['split_counts'])
print('All corpus and question hashes match; all supporting absence checks passed.')

Frozen questions: 30
Split counts: {'train': 23, 'test': 7}
All corpus and question hashes match; all supporting absence checks passed.


## Verify the Git freeze

The committed bytes must match the local question set and manifest. This check prevents accidentally evaluating an edited set. Commit history records the freeze before the improvement loop.

In [2]:
def git(*args):
    return subprocess.check_output(['git', '-C', str(ROOT), *args])

for relative in ['evaluation/unanswerable.json', 'evaluation/frozen_manifest.json']:
    assert git('show', f'HEAD:{relative}') == (ROOT / relative).read_bytes(), f'Commit {relative} first.'
freeze_commit = git('log', '-1', '--format=%H', '--', 'evaluation/unanswerable.json').decode().strip()
print('Question-set commit:', freeze_commit)
train_negatives = [r for r in records if r['split'] == 'train']
display(pd.DataFrame(train_negatives)[['query', 'category', 'reason']])

Question-set commit: 52fc9556ef3ef13d5ffc2718c2586cfb6e863945


,query,category,reason
0,Consider the distribution agreement between Ni...,missing_agreement,The named agreement and fictional counterparti...
1,Consider the co-production agreement between C...,missing_agreement,The named agreement and fictional counterparti...
2,Consider the sponsorship agreement between Lum...,missing_agreement,The named agreement and fictional counterparti...
3,Consider the content license between QuartzMea...,missing_agreement,The named agreement and fictional counterparti...
4,Consider the distribution agreement between Wi...,missing_agreement,The named agreement and fictional counterparti...
5,Consider the promotional rights agreement betw...,missing_agreement,The named agreement and fictional counterparti...
6,Consider the merchandising license between Ced...,missing_agreement,The named agreement and fictional counterparti...
7,Consider the Wireless Content License Agreemen...,unsupported_provision,"The named source is indexed, but the requested..."
8,Consider the Jacksonville Jaguars sponsorship ...,unsupported_provision,"The named source is indexed, but the requested..."
9,Consider the service agreement between Bicycle...,unsupported_provision,"The named source is indexed, but the requested..."


## Evaluate the unchanged baseline

Use the same BM25 configuration and positive-score rule as step 2. Do not tune the threshold in this checkpoint. A matched word is not evidence of an answer, so false returns are expected.

**Correct abstention:** returns nothing for an unanswerable question. **False return:** returns one or more passages despite the negative label. Step 2 separately measured false abstention on answerable train questions. These two rates must be considered together.

In [3]:
config = json.loads((ROOT / 'configs/bm25_baseline.json').read_text(encoding='utf-8'))
retriever = BM25Retriever(documents, config)
result = evaluate_negative_train(retriever, train_negatives, k=5)
print(f"Correct abstention: {result['correct_abstentions']}/{result['n_queries']} ({result['correct_abstention_rate']:.2%})")
print(f"False returns: {result['false_return_rate']:.2%}")
summary = pd.DataFrame(result['predictions']).groupby('category')['abstained'].agg(['count', 'sum', 'mean'])
display(summary.rename(columns={'count':'n_queries', 'sum':'correct_abstentions', 'mean':'correct_abstention_rate'}))

Correct abstention: 0/23 (0.00%)
False returns: 100.00%


,n_queries,correct_abstentions,correct_abstention_rate
category,,,
missing_agreement,7,0,0.0
subsequent_event,8,0,0.0
unsupported_provision,8,0,0.0


## Save and inspect train results

Save a new run without changing the frozen questions or the original retrieval baseline. The run links the corpus, labeling commit, configuration, and code hashes. No test retrieval or test abstention rate is computed.

In [4]:
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = ROOT / 'results' / 'bm25_abstention' / run_id
output.mkdir(parents=True, exist_ok=False)
metadata = dict(run_id=run_id, split='train', question_set_commit=freeze_commit,
                frozen_manifest=frozen, config=config, test_evaluation_runs=0,
                code_sha256={name: hashlib.sha256((ROOT / name).read_bytes()).hexdigest()
                             for name in ['retrieval.py', 'negative_validation.py']})
(output / 'run.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
(output / 'results.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
summary.to_csv(output / 'by_category.csv')
print('Saved:', output)
failed = [p for p in result['predictions'] if not p['abstained']]
if failed:
    print('Example false return:', failed[0]['query'])
    display(pd.DataFrame(failed[0]['hits'][:3])[['score','file_path','start','end','text']])

Saved: C:\Users\vatsa\OneDrive\Desktop\JioStar\results\bm25_abstention\20261101T020853204883Z
Example false return: Consider the distribution agreement between NimbleCove Media and BluePeak Distribution; Does the agreement allow sublicensing to third-party platforms?


,score,file_path,start,end,text
0,31.239660,"cuad/DUOSTECHNOLOGIESGROUP,INC_04_21_2009-EX-1...",7200,8200,e Documentation. Rubicon shall promptly notif...
1,27.140007,cuad/BONTONSTORESINC_04_20_2018-EX-99.3-AGENCY...,158400,159400,"tores and the Distribution Centers, the Headqu..."
2,26.273197,cuad/BONTONSTORESINC_04_20_2018-EX-99.3-AGENCY...,50400,51400,gent shall be responsible for allocating and d...


## Explain this step

We froze a separate set of questions whose answers are unavailable in the corpus. It tests whether keyword retrieval can distinguish relevant answers from passages that merely share words.

The set is small and synthetic, with only seven test negatives and repeated topic families. Report counts alongside rates; results cannot establish production reliability. Future external validation should include subtle absent clauses and realistic legal near-misses, without editing this frozen set during the improvement loop.

**Next:** declare the primary retrieval objective and abstention/precision guardrails, then improve the baseline using train data only. No LLM calls occur here. When LLMs are introduced, credentials stay in environment variables and evaluation must support replay from a versioned output cache.